# OODimprovements 08: Multi-Scale Gaussian TTA & Sliced Inference Engine
Test-Time Deployment Engine implementing Phase 0 from `next_moves_forOOD.md`:
- **Multi-Scale Gaussian TTA**: Evaluates uncropped megapixel photos ($2000 \times 1500$) across multi-scale patches ($640 \times 640$ and $768 \times 768$, $25\%$ overlap) with **2D Gaussian Apodization Blending**.
- **Head-to-Head Comparison Across 3 Paradigms**:
  1. Direct Resizing ($2000 \times 1500 \to 640 \times 640$)
  2. Single-Scale Gaussian Tiling ($640 \times 640$ patches, 25% overlap, Gaussian apodization)
  3. Multi-Scale Gaussian TTA ($640 \times 640$ + $768 \times 768$ fused Gaussian apodization)
- **Checkpoints**: Automatically discovers and benchmarks all available checkpoints in `/kaggle/input` (e.g. `03_mosaic_native`, `05_twostage`, `06_res_preserving`, `07_tversky`, or prior models).
- **Outputs**: Comprehensive metrics table (Dice, Precision, Recall, IoU) and JSON export.


In [ ]:
# -- Environment & Imports --
!mkdir -p scripts configs utils distillation data/datasets results repacked_ckpts
!pip install -q ultralytics albumentations pycocotools opencv-python Pillow matplotlib tqdm pandas
import os, cv2, json, time, glob, zipfile, shutil
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm
from ultralytics import YOLO
import torch


In [ ]:
%%writefile scripts/convert_crack500_uncropped.py
#!/usr/bin/env python3
"""
Crack500 Uncropped Test/Val → YOLO seg format converter
======================================================
Converts the original uncropped validation and test sets of Crack500.
Handles EXIF orientation for images by rotating the corresponding masks.

Source directories:
  data/datasets/crack500/valdata/   ← contains {stem}.jpg and {stem}_mask.png
  data/datasets/crack500/testdata/  ← contains {stem}.jpg and {stem}_mask.png

Output (YOLO seg format):
  data/datasets/crack500_uncropped_yolo/
  ├── images/
  │   ├── val/
  │   └── test/
  ├── labels/
  │   ├── val/
  │   └── test/
  └── dataset.yaml
"""

import os
import cv2
import numpy as np
import argparse
import shutil
from pathlib import Path
from tqdm import tqdm
from PIL import Image


CLASS_ID = 0        # single class: crack
MIN_AREA = 50       # minimum pixel area to keep an instance
MIN_POINTS = 6      # minimum polygon points (3 coordinate pairs)


def get_exif_rotation(img_path: Path):
    """Retrieve EXIF orientation tag from image."""
    try:
        with Image.open(img_path) as im:
            exif = im.getexif()
            if exif:
                return exif.get(274)  # 274 is the Orientation tag
    except Exception:
        pass
    return None


def rotate_mask_to_match_image(mask: np.ndarray, exif_orientation: int) -> np.ndarray:
    """Rotate mask array to match image rotation applied by cv2.imread based on EXIF."""
    if exif_orientation == 6:
        return cv2.rotate(mask, cv2.ROTATE_90_CLOCKWISE)
    elif exif_orientation == 8:
        return cv2.rotate(mask, cv2.ROTATE_90_COUNTERCLOCKWISE)
    elif exif_orientation == 3:
        return cv2.rotate(mask, cv2.ROTATE_180)
    return mask


def binary_mask_to_yolo_instances(mask_path: str, img_w: int, img_h: int, exif_orientation: int = None) -> list[str]:
    """
    Read binary PNG mask → rotate based on EXIF → split into instances via connectedComponents
    → convert each to normalized YOLO seg polygon string.
    """
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    if mask is None:
        return []

    if exif_orientation:
        mask = rotate_mask_to_match_image(mask, exif_orientation)

    # Threshold (Crack500 masks are binary 0/255)
    binary = (mask > 127).astype(np.uint8)

    # Separate touching cracks into individual instances
    num_labels, labels_map = cv2.connectedComponents(binary)

    label_lines = []
    for label_id in range(1, num_labels):      # 0 = background
        instance = (labels_map == label_id).astype(np.uint8)

        if instance.sum() < MIN_AREA:
            continue

        # Find contours for this instance
        contours, _ = cv2.findContours(
            instance, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
        )

        for contour in contours:
            if len(contour) < MIN_POINTS // 2:
                continue

            # Flatten and normalize to [0, 1]
            pts = contour.squeeze()
            if pts.ndim == 1:
                pts = pts.reshape(1, 2)

            # Simplify contour slightly to reduce file size
            epsilon = 0.002 * cv2.arcLength(contour, True)
            simplified = cv2.approxPolyDP(contour, epsilon, True).squeeze()
            if simplified.ndim == 1:
                simplified = simplified.reshape(1, 2)
            if len(simplified) < 3:
                simplified = pts

            norm = []
            for x, y in simplified:
                norm.append(x / img_w)
                norm.append(y / img_h)

            if len(norm) < MIN_POINTS:
                continue

            coords_str = " ".join(f"{v:.6f}" for v in norm)
            label_lines.append(f"{CLASS_ID} {coords_str}")

    return label_lines


def process_split(src_dir: Path, dst_dir: Path, split_name: str):
    """Process uncropped val or test split."""
    split_dir = src_dir / f"{split_name}data"
    if not split_dir.exists():
        print(f"  [Warning] Directory {split_dir} does not exist, skipping split {split_name}.")
        return 0

    dst_img_dir = dst_dir / "images" / split_name
    dst_lbl_dir = dst_dir / "labels" / split_name

    dst_img_dir.mkdir(parents=True, exist_ok=True)
    dst_lbl_dir.mkdir(parents=True, exist_ok=True)

    # Find all image files (jpg/jpeg/png that do not contain '_mask')
    all_files = sorted(split_dir.iterdir())
    image_files = [
        f for f in all_files 
        if f.suffix.lower() in ('.jpg', '.jpeg', '.png')
        and '_mask' not in f.name.lower()
        and ':Zone.Identifier' not in f.name
    ]

    converted = 0
    skipped = 0

    for img_path in tqdm(image_files, desc=f"  {split_name}", leave=False):
        stem = img_path.stem

        # Find mask (stem + "_mask.png")
        mask_path = split_dir / f"{stem}_mask.png"
        if not mask_path.exists():
            skipped += 1
            continue

        # Read image to get dimensions (matches how cv2.imread auto-rotates it based on EXIF)
        img = cv2.imread(str(img_path))
        if img is None:
            skipped += 1
            continue
        h, w = img.shape[:2]

        # Get EXIF rotation from image
        exif_orientation = get_exif_rotation(img_path)

        # Convert mask to YOLO seg labels (rotating it to match)
        label_lines = binary_mask_to_yolo_instances(str(mask_path), w, h, exif_orientation)

        # Copy image
        dst_img_path = dst_img_dir / img_path.name
        shutil.copy2(img_path, dst_img_path)

        # Write label file (even if empty — YOLO needs it)
        dst_lbl_path = dst_lbl_dir / f"{stem}.txt"
        with open(dst_lbl_path, "w") as f:
            f.write("\n".join(label_lines))

        converted += 1

    print(f"  {split_name}: {converted} images converted, {skipped} skipped")
    return converted


def main():
    parser = argparse.ArgumentParser(description="Convert Crack500 Uncropped splits to YOLO seg format")
    parser.add_argument(
        "--src",
        type=str,
        default="data/datasets/crack500",
        help="Path to crack500 root dir"
    )
    parser.add_argument(
        "--dst",
        type=str,
        default="data/datasets/crack500_uncropped_yolo",
        help="Output directory"
    )
    args = parser.parse_args()

    src = Path(args.src).expanduser().resolve()
    dst = Path(args.dst).expanduser().resolve()

    print(f"[Convert] Source: {src}")
    print(f"[Convert] Output: {dst}")
    print()

    if not src.exists():
        print(f"ERROR: Source directory not found: {src}")
        return

    if dst.exists():
        print(f"[Warning] Output directory exists, clearing: {dst}")
        shutil.rmtree(dst)

    dst.mkdir(parents=True, exist_ok=True)

    counts = {}
    for split in ["val", "test"]:
        n = process_split(src, dst, split)
        counts[split] = n

    # Write dataset_uncropped.yaml
    yaml_content = f"""# Crack500 Uncropped — YOLO seg format
# Auto-generated by convert_crack500_uncropped.py

path: {dst.resolve()}
train: images/val
val:   images/val
test:  images/test

nc: 1
names:
  0: crack

# Stats
# val:   ~{counts.get('val', 0)} images (uncropped)
# test:  ~{counts.get('test', 0)} images (uncropped)
"""
    with open(dst / "dataset.yaml", "w") as f:
        f.write(yaml_content)
    print(f"\n  dataset.yaml written to {dst / 'dataset.yaml'}")
    print(f"[Done] Converted uncropped splits successfully.")


if __name__ == "__main__":
    main()


In [ ]:
# -- Step 1: Link or Convert Uncropped Validation Dataset --
input_dir = Path("/kaggle/input/distill_datasetforme")
if not input_dir.exists():
    input_dir = Path("/kaggle/input")

for root, dirs, files in os.walk(str(input_dir)):
    root_p = Path(root)
    if "valdata" in dirs or "testdata" in dirs:
        !python scripts/convert_crack500_uncropped.py --src {root_p} --dst data/datasets/crack500_uncropped_yolo
        break

print("Uncropped validation dataset ready at data/datasets/crack500_uncropped_yolo/")


In [ ]:
# -- Step 2: Multi-Scale Gaussian TTA Inference Engine --
def create_gaussian_weight_map(tile_size=640, sigma=0.35):
    """Generates a 2D Gaussian window to smoothly blend overlapping tiles."""
    ax = np.linspace(-1, 1, tile_size)
    gauss_1d = np.exp(-0.5 * (ax / sigma) ** 2)
    gauss_2d = np.outer(gauss_1d, gauss_1d).astype(np.float32)
    gauss_2d = np.maximum(gauss_2d, 0.05)  # baseline floor for borders
    return gauss_2d / gauss_2d.max()


def predict_sliding_window_gaussian(model, img_bgr, tile_size=640, overlap=0.25, conf=0.25, sigma=0.35):
    """Single-scale tiled inference with 2D Gaussian apodization blending."""
    h, w = img_bgr.shape[:2]
    stride = int(tile_size * (1.0 - overlap))
    full_prob_map = np.zeros((h, w), dtype=np.float32)
    weight_accum_map = np.zeros((h, w), dtype=np.float32)
    weight_window = create_gaussian_weight_map(tile_size, sigma)

    y_steps = list(range(0, max(1, h - tile_size + 1), stride))
    if len(y_steps) == 0 or (y_steps[-1] + tile_size < h):
        y_steps.append(max(0, h - tile_size))
        
    x_steps = list(range(0, max(1, w - tile_size + 1), stride))
    if len(x_steps) == 0 or (x_steps[-1] + tile_size < w):
        x_steps.append(max(0, w - tile_size))

    for y0 in y_steps:
        for x0 in x_steps:
            tile = img_bgr[y0:y0+tile_size, x0:x0+tile_size]
            actual_th, actual_tw = tile.shape[:2]
            if actual_th != tile_size or actual_tw != tile_size:
                tile_padded = np.zeros((tile_size, tile_size, 3), dtype=np.uint8)
                tile_padded[:actual_th, :actual_tw] = tile
                tile = tile_padded

            results = model.predict(tile, imgsz=tile_size, conf=conf, verbose=False, device="cuda" if torch.cuda.is_available() else "cpu")[0]
            tile_prob = np.zeros((tile_size, tile_size), dtype=np.float32)

            if results.masks is not None and len(results.masks) > 0:
                for m in results.masks.data.cpu().numpy():
                    m_resized = cv2.resize(m, (tile_size, tile_size))
                    tile_prob = np.maximum(tile_prob, m_resized)

            actual_h = min(tile_size, h - y0)
            actual_w = min(tile_size, w - x0)

            full_prob_map[y0:y0+actual_h, x0:x0+actual_w] += tile_prob[:actual_h, :actual_w] * weight_window[:actual_h, :actual_w]
            weight_accum_map[y0:y0+actual_h, x0:x0+actual_w] += weight_window[:actual_h, :actual_w]

    weight_accum_map = np.maximum(weight_accum_map, 1e-5)
    full_prob_map = full_prob_map / weight_accum_map
    return (full_prob_map > 0.35).astype(np.uint8)


def predict_multiscale_gaussian_tta(model, img_bgr, tile_sizes=(640, 768), overlap=0.25, conf=0.25, sigma=0.35):
    """Multi-scale test-time augmentation across tile sizes (e.g. 640 + 768)."""
    h, w = img_bgr.shape[:2]
    fused_prob = np.zeros((h, w), dtype=np.float32)

    for ts in tile_sizes:
        prob_map = np.zeros((h, w), dtype=np.float32)
        weight_accum = np.zeros((h, w), dtype=np.float32)
        weight_window = create_gaussian_weight_map(ts, sigma)
        stride = int(ts * (1.0 - overlap))

        y_steps = list(range(0, max(1, h - ts + 1), stride))
        if len(y_steps) == 0 or (y_steps[-1] + ts < h):
            y_steps.append(max(0, h - ts))
        x_steps = list(range(0, max(1, w - ts + 1), stride))
        if len(x_steps) == 0 or (x_steps[-1] + ts < w):
            x_steps.append(max(0, w - ts))

        for y0 in y_steps:
            for x0 in x_steps:
                tile = img_bgr[y0:y0+ts, x0:x0+ts]
                actual_th, actual_tw = tile.shape[:2]
                if actual_th != ts or actual_tw != ts:
                    tile_padded = np.zeros((ts, ts, 3), dtype=np.uint8)
                    tile_padded[:actual_th, :actual_tw] = tile
                    tile = tile_padded

                results = model.predict(tile, imgsz=ts, conf=conf, verbose=False, device="cuda" if torch.cuda.is_available() else "cpu")[0]
                tile_prob = np.zeros((ts, ts), dtype=np.float32)

                if results.masks is not None and len(results.masks) > 0:
                    for m in results.masks.data.cpu().numpy():
                        m_resized = cv2.resize(m, (ts, ts))
                        tile_prob = np.maximum(tile_prob, m_resized)

                actual_h = min(ts, h - y0)
                actual_w = min(ts, w - x0)
                prob_map[y0:y0+actual_h, x0:x0+actual_w] += tile_prob[:actual_h, :actual_w] * weight_window[:actual_h, :actual_w]
                weight_accum[y0:y0+actual_h, x0:x0+actual_w] += weight_window[:actual_h, :actual_w]

        prob_map = prob_map / np.maximum(weight_accum, 1e-5)
        fused_prob += prob_map

    fused_prob = fused_prob / len(tile_sizes)
    return (fused_prob > 0.35).astype(np.uint8)

print("Multi-Scale Gaussian TTA Inference Engine ready!")


In [ ]:
# -- Step 3: Discover Checkpoints & Run Cross-Evaluation --
from PIL import Image

def get_exif_rotation(img_path: Path):
    """Retrieve EXIF orientation tag from image."""
    try:
        with Image.open(img_path) as im:
            exif = im.getexif()
            if exif:
                return exif.get(274)  # 274 is the Orientation tag
    except Exception:
        pass
    return None

def rotate_mask_to_match_image(mask: np.ndarray, exif_orientation: int) -> np.ndarray:
    """Rotate mask array to match image rotation applied by cv2.imread based on EXIF."""
    if exif_orientation == 6:
        return cv2.rotate(mask, cv2.ROTATE_90_CLOCKWISE)
    elif exif_orientation == 8:
        return cv2.rotate(mask, cv2.ROTATE_90_COUNTERCLOCKWISE)
    elif exif_orientation == 3:
        return cv2.rotate(mask, cv2.ROTATE_180)
    return mask

def compute_metrics(pred_mask, gt_mask):
    # Defensive shape safeguard: guarantee identical broadcast dimensions under any condition
    if pred_mask.shape != gt_mask.shape:
        gt_mask = cv2.resize(gt_mask.astype(np.uint8), (pred_mask.shape[1], pred_mask.shape[0]), interpolation=cv2.INTER_NEAREST)

    intersection = np.logical_and(pred_mask > 0, gt_mask > 0).sum()
    total = (pred_mask > 0).sum() + (gt_mask > 0).sum()
    pred_sum = (pred_mask > 0).sum()
    gt_sum = (gt_mask > 0).sum()

    dice = float(2.0 * intersection / total) if total > 0 else (1.0 if intersection == 0 else 0.0)
    precision = float(intersection / pred_sum) if pred_sum > 0 else (1.0 if gt_sum == 0 else 0.0)
    recall = float(intersection / gt_sum) if gt_sum > 0 else (1.0 if pred_sum == 0 else 0.0)
    union = pred_sum + gt_sum - intersection
    iou = float(intersection / union) if union > 0 else 1.0

    return {"dice": dice, "precision": precision, "recall": recall, "iou": iou}

repack_dir = Path("/kaggle/working/repacked_ckpts")
repack_dir.mkdir(parents=True, exist_ok=True)

# A. Handle unzipped PyTorch model folders (contains data.pkl like bestmosaic/best)
if os.path.exists("/kaggle/input"):
    for root, dirs, files in os.walk("/kaggle/input"):
        if "data.pkl" in files:
            parts = [p for p in Path(root).parts if p not in [".", ""]]
            folder_name = Path(root).name
            if folder_name in ["best", "weights", "model", "archive", "default", "1", ".", ""]:
                skip_set = {"kaggle", "input", "models", "pytorch", "default", "1", "best", "weights", "model", "archive"}
                meaningful = [p for p in parts if p not in skip_set]
                folder_name = meaningful[-1] if meaningful else Path(root).parent.name
            repacked_path = repack_dir / f"{folder_name}.pt"
            print(f"  -> Detected unzipped PyTorch folder: {root}")
            print(f"  -> Repacking into valid PyTorch container: {repacked_path}...")
            with zipfile.ZipFile(repacked_path, 'w', compression=zipfile.ZIP_STORED) as zf:
                for r, d, f_list in os.walk(root):
                    for file_name in f_list:
                        full_file = os.path.join(r, file_name)
                        rel_file = os.path.relpath(full_file, root)
                        arc_name = os.path.join("archive", rel_file).replace("\\", "/")
                        zf.write(full_file, arc_name)
            print(f"  -> Successfully created valid model: {repacked_path}")

# B. Scan checkpoints across repacked, input, working, and runs directories
checkpoints = {}
search_roots = ["/kaggle/working/repacked_ckpts", "/kaggle/input", "/kaggle/working", "runs"]
stock_weights = {"yolo11n-seg.pt", "yolov8n-seg.pt", "yolo11s-seg.pt", "yolo11m-seg.pt"}

for s_root in search_roots:
    if os.path.exists(s_root):
        for root, dirs, files in os.walk(s_root):
            for f in files:
                if (f.endswith(".pt") or f.endswith(".pth")) and "sam" not in f.lower() and "hiera" not in f.lower():
                    if f in stock_weights and "runs" not in root:
                        continue
                    resolved = str(Path(os.path.join(root, f)).resolve())
                    p = Path(resolved)
                    if p.parent.name == "weights":
                        exp_name = p.parent.parent.name
                        tag = f"{exp_name}_{p.stem}"
                    elif p.parent.name == "repacked_ckpts":
                        tag = p.stem
                    else:
                        parent_name = p.parent.name
                        tag = f"{parent_name}_{p.stem}" if parent_name not in ["working", "input"] else p.stem
                    
                    base_tag = tag
                    c_idx = 1
                    while tag in checkpoints and checkpoints[tag] != resolved:
                        tag = f"{base_tag}_{c_idx}"
                        c_idx += 1

                    if tag not in checkpoints:
                        checkpoints[tag] = resolved

if not checkpoints:
    print("[Warning] No finetuned checkpoints found; downloading stock yolo11n-seg.pt as fallback")
    checkpoints["yolo11n-seg_baseline"] = "yolo11n-seg.pt"

print(f"Discovered {len(checkpoints)} checkpoints to evaluate:")
for k, v in checkpoints.items():
    print(f"  * {k}: {v}")

# C. Pre-index ground-truth masks across /kaggle/input, /kaggle/working, and data/
print("\n[Step 3] Indexing ground-truth mask database...")
mask_database = {}
for search_dir in ["/kaggle/input", "/kaggle/working", "data"]:
    if os.path.exists(search_dir):
        for root, dirs, files in os.walk(search_dir):
            if "teacher_logits" in root:
                continue
            for f in files:
                if f.endswith("_mask.png") or (f.endswith(".png") and not f.endswith("_prob.png")):
                    clean_stem = f.replace("_mask.png", "").replace(".png", "")
                    mask_database[clean_stem] = os.path.join(root, f)

print(f"  -> Indexed {len(mask_database)} ground-truth mask files.")

# D. Load test pairs
val_dir = Path("data/datasets/crack500_uncropped_yolo")
img_files = sorted(list((val_dir / "images/val").glob("*.jpg")) + list((val_dir / "images/val").glob("*.png")))
if not img_files:
    img_files = sorted(list((val_dir / "images/test").glob("*.jpg")) + list((val_dir / "images/test").glob("*.png")))

test_pairs = []
for img_p in img_files:
    stem = img_p.stem
    if stem in mask_database:
        test_pairs.append((img_p, Path(mask_database[stem])))

print(f"Prepared {len(test_pairs)} uncropped evaluation photo-mask pairs.")

results = {}
for ckpt_name, ckpt_path in checkpoints.items():
    print(f"\nEvaluating checkpoint: {ckpt_name}...")
    try:
        model = YOLO(ckpt_path)
    except Exception as e:
        print(f"  Failed to load {ckpt_name}: {e}")
        continue

    dices_direct = []
    dices_tiled = []
    dices_tta = []
    precisions_tta = []
    recalls_tta = []

    for img_p, mask_p in tqdm(test_pairs[:50], desc=f"Eval {ckpt_name}"):  # evaluate first 50 for speed
        img = cv2.imread(str(img_p))
        gt = cv2.imread(str(mask_p), cv2.IMREAD_GRAYSCALE)
        if img is None or gt is None:
            continue
        h, w = img.shape[:2]

        # 1. Align EXIF orientation between auto-oriented JPEG and raw PNG mask
        exif_rot = get_exif_rotation(img_p)
        if exif_rot:
            gt = rotate_mask_to_match_image(gt, exif_rot)

        # 2. Geometric transpose check (e.g. 1440x2560 vs 2560x1440)
        if gt.shape[:2] == (w, h):
            gt = cv2.rotate(gt, cv2.ROTATE_90_CLOCKWISE)

        # 3. Final dimension safety resize
        if gt.shape[:2] != (h, w):
            gt = cv2.resize(gt, (w, h), interpolation=cv2.INTER_NEAREST)

        gt_bin = (gt > 127).astype(np.uint8)

        # 1. Direct Resize (640)
        res_dir = model.predict(img, imgsz=640, conf=0.25, verbose=False, device="cuda" if torch.cuda.is_available() else "cpu")[0]
        pred_direct = np.zeros((h, w), dtype=np.uint8)
        if res_dir.masks is not None and len(res_dir.masks) > 0:
            pred_dir_mask = (res_dir.masks.data.cpu().numpy().max(axis=0) > 0.5).astype(np.uint8)
            pred_direct = cv2.resize(pred_dir_mask, (w, h), interpolation=cv2.INTER_NEAREST)
        dices_direct.append(compute_metrics(pred_direct, gt_bin)["dice"])

        # 2. Gaussian Tiled (640)
        pred_tiled = predict_sliding_window_gaussian(model, img, tile_size=640, overlap=0.25, conf=0.25)
        dices_tiled.append(compute_metrics(pred_tiled, gt_bin)["dice"])

        # 3. Multi-Scale Gaussian TTA (640 + 768)
        pred_tta = predict_multiscale_gaussian_tta(model, img, tile_sizes=(640, 768), overlap=0.25, conf=0.25)
        m_tta = compute_metrics(pred_tta, gt_bin)
        dices_tta.append(m_tta["dice"])
        precisions_tta.append(m_tta["precision"])
        recalls_tta.append(m_tta["recall"])

    results[ckpt_name] = {
        "mean_dice_direct": float(np.mean(dices_direct)) if dices_direct else 0.0,
        "mean_dice_tiled_640": float(np.mean(dices_tiled)) if dices_tiled else 0.0,
        "mean_dice_multiscale_tta": float(np.mean(dices_tta)) if dices_tta else 0.0,
        "mean_precision_tta": float(np.mean(precisions_tta)) if precisions_tta else 0.0,
        "mean_recall_tta": float(np.mean(recalls_tta)) if recalls_tta else 0.0,
    }
    print(f"  -> Direct Dice: {results[ckpt_name]['mean_dice_direct']:.4f}")
    print(f"  -> Tiled 640 Dice: {results[ckpt_name]['mean_dice_tiled_640']:.4f}")
    print(f"  -> Multi-Scale TTA Dice: {results[ckpt_name]['mean_dice_multiscale_tta']:.4f}")
    print(f"  -> TTA Precision: {results[ckpt_name]['mean_precision_tta']:.4f} | Recall: {results[ckpt_name]['mean_recall_tta']:.4f}")

# Save results JSON
out_path = Path("/kaggle/working/results/multiscale_tta_sahi_eval_summary.json")
out_path.parent.mkdir(parents=True, exist_ok=True)
with open(out_path, "w") as f:
    json.dump(results, f, indent=2)

print(f"\nEvaluation summary saved to {out_path}!")
